# MED PAL-on-Spark: Databricks Free Edition parity + Delta demo

Same 3-scenario parity assertions as tests_spark (adapted to the provided serverless session - no local SparkSession.builder), validation-raise checks, and a Delta round-trip + append on a UC volume.

> **Archival reconstruction, documentation only (note added for this public copy).** This notebook documents one completed run on 2026-11-14. It is not a native Databricks export: the code cells are the exact imported source (with identifiers redacted) and the outputs were captured from the rendered notebook and embedded afterward. This copy was NOT executed after redaction, and the `<databricks-user>` placeholder paths are non-functional. Do not re-run it: re-running the Delta cell against an existing output path can append rows before the count assertion fails, and the existing table persists. It is documentation of a run, not a runnable notebook.

In [ ]:
import os, zipfile

PROJECT_ROOT = "/Workspace/Users/<databricks-user>/med-pal"

cats = [r[0] for r in spark.sql("SHOW CATALOGS").collect()]
print("catalogs:", cats)
catalog = "workspace" if "workspace" in cats else [c for c in cats if not c.startswith("system")][0]
ZIP_PATH = "/Volumes/%s/default/med_pal_bundle/1-bundle.zip" % catalog
print("bundle zip:", ZIP_PATH)

os.makedirs(PROJECT_ROOT, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall(PROJECT_ROOT)

count = 0
for root, dirs, files in os.walk(PROJECT_ROOT):
    dirs[:] = [d for d in dirs if d != "__pycache__"]
    for n in sorted(files):
        p = os.path.join(root, n)
        count += 1
        print("%8d  %s" % (os.path.getsize(p), os.path.relpath(p, PROJECT_ROOT)))
print("extracted %d files to %s" % (count, PROJECT_ROOT))


catalogs: ['samples', 'system', 'workspace']
bundle zip: /Volumes/workspace/default/med_pal_bundle/1-bundle.zip
     611  pyproject.toml
     789  requirements-spark.txt
     377  requirements.txt
    2268  med_spark/README.md
     365  med_spark/__init__.py
    9851  med_spark/batch.py
    1871  med_spark/delta_store.py
    4254  med_spark/demo_batch.py
      61  tests_spark/__init__.py
     340  tests_spark/conftest.py
    6785  tests_spark/test_batch_spark.py
     943  med/__init__.py
    2459  med/config.py
    1995  med/demo.py
    3690  med/pipeline.py
     529  med/provenance.py
    2717  med/types.py
      57  med/storage/__init__.py
    4422  med/storage/sqlite_store.py
      47  med/data/__init__.py
    4739  med/data/synthetic.py
    2038  med/data/yfinance_source.py
      64  med/agents/__init__.py
    2809  med/agents/contextualist.py
    1916  med/agents/decision_maker.py
    4159  med/agents/statistician.py
      47  tests/__init__.py
     340  tests/conftest.py
    1178

In [ ]:
import platform
import pandas as pd
import pyarrow

spark.conf.set("spark.sql.session.timeZone", "UTC")
print("python:", platform.python_version())
print("spark:", spark.version)
print("pandas:", pd.__version__)
print("pyarrow:", pyarrow.__version__)
print("session tz:", spark.conf.get("spark.sql.session.timeZone"))


python: 3.11.10
spark: 4.2.0
pandas: 1.5.3
pyarrow: 14.0.1
session tz: UTC


In [ ]:
import math
import sys

sys.path.insert(0, "/Workspace/Users/<databricks-user>/med-pal")

import pandas as pd
from pyspark.sql import functions as F

from med.config import MedConfig
from med.data.synthetic import build_demo_scenarios
from med.pipeline import Pipeline
from med_spark.batch import compute_pal_events
from med_spark.demo_batch import scenario_frames

config = MedConfig()
scenarios = {s.name: s for s in build_demo_scenarios(config)}
events, bars = scenario_frames(spark, config=config)
computed = compute_pal_events(events, bars, config=config).collect()

def aware_utc(v):
    if v is None:
        return None
    t = pd.Timestamp(v)
    return (t.tz_localize("UTC") if t.tzinfo is None else t.tz_convert("UTC")).to_pydatetime()

def close(a, b, tol=1e-6):
    if a is None or b is None:
        return a is None and b is None
    return math.isclose(float(a), float(b), rel_tol=1e-9, abs_tol=tol)

for name in ["fast", "slow", "no_event"]:
    sc = scenarios[name]
    expected = Pipeline(config=config, store=None).run(
        sc.news_event, sc.bars, tier=sc.tier.value, seed=sc.seed
    ).pal_event
    rows = [r for r in computed if r["event_id"] == sc.news_event.event_id]
    assert len(rows) == 1, name
    row = rows[0]
    assert row["event_id"] == expected.event_id
    assert row["ticker"] == expected.ticker
    assert row["tier"] == expected.tier.value
    assert aware_utc(row["t_event"]) == aware_utc(expected.t_event)
    assert aware_utc(row["t_efficient"]) == aware_utc(expected.t_efficient)
    assert close(row["pal_seconds"], expected.pal_seconds)
    assert row["pal_class"] == expected.pal_class.value
    assert math.isclose(row["z_score"], expected.z_score, rel_tol=1e-7, abs_tol=1e-9)
    assert math.isclose(row["sigma_pre"], expected.sigma_pre, rel_tol=1e-7, abs_tol=1e-9)
    assert close(row["equilibrium_price"], expected.equilibrium_price, 1e-9)
    assert math.isclose(row["sentiment_score"], expected.sentiment_score, rel_tol=1e-7, abs_tol=1e-9)
    assert row["event_type"] == expected.event_type.value
    assert row["news_weight"] == expected.news_weight.value
    assert row["data_vintage"] == expected.data_vintage
    assert row["generator_seed"] == expected.generator_seed
    if row["git_sha"] is not None and expected.git_sha is not None:
        assert row["git_sha"] == expected.git_sha
    print("PASS parity[%s]  event_id=%s class=%s pal_seconds=%s" % (name, row["event_id"], row["pal_class"], row["pal_seconds"]))

# Validation raises (mirror tests_spark raise-tests)
def expect_raise(label, pattern, fn):
    try:
        fn()
    except ValueError as e:
        assert pattern in str(e), (label, str(e))
        print("PASS raise[%s]  (%s)" % (label, pattern))
        return
    raise AssertionError(label + " did not raise")

dup = events.withColumn("event_id", F.lit("duplicate-id"))
expect_raise("duplicate_event_id", "duplicate event_id", lambda: compute_pal_events(dup, bars, config=config).count())

extra_ev = events.filter(F.col("scenario") == "fast").withColumn("event_id", F.lit("fast-extra"))
expect_raise("two_events_one_scenario", "more than one event row", lambda: compute_pal_events(events.unionByName(extra_ev), bars, config=config).count())

extra_bars = bars.filter(F.col("scenario") == "fast").withColumn("scenario", F.lit("orphan"))
expect_raise("bars_only_scenario", "scenario sets differ", lambda: compute_pal_events(events, bars.unionByName(extra_bars), config=config).count())

nulled = events.withColumn("event_id", F.lit(None).cast("string"))
expect_raise("null_event_id", "null scenario/event_id/ts_published", lambda: compute_pal_events(nulled, bars, config=config).count())


PASS parity[fast]  event_id=fast-1 class=FAST pal_seconds=120.0
PASS parity[slow]  event_id=slow-2 class=SLOW pal_seconds=2580.0
PASS parity[no_event]  event_id=no_event-3 class=INCONCLUSIVE pal_seconds=0.0
PASS raise[duplicate_event_id] (duplicate event_id)
PASS raise[two_events_one_scenario] (more than one event row)
PASS raise[bars_only_scenario] (scenario sets differ)
PASS raise[null_event_id] (null scenario/event_id/ts_published)


In [ ]:
from med_spark.batch import compute_pal_events, PAL_COLUMNS
from med_spark.delta_store import DeltaPalStore
from med_spark.demo_batch import scenario_frames

events, bars = scenario_frames(spark, config=config)
computed = compute_pal_events(events, bars, config=config)

cats = [r[0] for r in spark.sql("SHOW CATALOGS").collect()]
print("catalogs:", cats)
catalog = "workspace" if "workspace" in cats else [c for c in cats if not c.startswith("system")][0]
spark.sql("CREATE SCHEMA IF NOT EXISTS %s.default" % catalog)
spark.sql("CREATE VOLUME IF NOT EXISTS %s.default.med_pal_delta" % catalog)
base = "/Volumes/%s/default/med_pal_delta/pal_events" % catalog
print("delta base path:", base)

store = DeltaPalStore(spark, base)
store.save(computed)
assert store.exists()
assert store.count() == events.count() == 3

written = {r["event_id"]: r for r in computed.collect()}
read_back = {r["event_id"]: r for r in store.list_all().collect()}
assert set(read_back) == set(written)
TS = ("t_event", "t_efficient")
FL = ("pal_seconds", "z_score", "sigma_pre", "equilibrium_price", "sentiment_score")
for eid, orig in written.items():
    rest = read_back[eid]
    for f in PAL_COLUMNS:
        if f in TS:
            assert aware_utc(rest[f]) == aware_utc(orig[f]), (eid, f)
        elif f in FL:
            assert close(rest[f], orig[f], 1e-12), (eid, f)
        else:
            assert rest[f] == orig[f], (eid, f)
print("PASS delta round-trip (16 contract fields x %d rows)" % len(written))

store.save(computed)
assert store.count() == 2 * events.count()
print("PASS delta append (count=%d)" % store.count())
store.list_all().show(truncate=False)


catalogs: ['samples', 'system', 'workspace']
delta base path: /Volumes/workspace/default/med_pal_delta/pal_events
PASS delta round-trip (16 contract fields x 3 rows)
PASS delta append (count=6)
+----------+------+----+-------------------+-------------------+-----------+------------+--------------------+---------------------+------------------+---------------+----------+-----------+--------------+--------------+-------+--------------------------+
|event_id  |ticker|tier|t_event            |t_efficient        |pal_seconds|pal_class   |z_score             |sigma_pre            |equilibrium_price |sentiment_score|event_type|news_weight|data_vintage  |generator_seed|git_sha|created_at                |
+----------+------+----+-------------------+-------------------+-----------+------------+--------------------+---------------------+------------------+---------------+----------+-----------+--------------+--------------+-------+--------------------------+
|slow-2    |ACME  |MID |2026-01-05 10:

In [ ]:
print("DBX-FE-DEMO-DONE")

DBX-FE-DEMO-DONE
